# Phase 12.8 — Search Failure Analysis

Load Phase 12.5 baseline artifacts and optional Phase 12.7 ranking experiment, classify judged-relevant failures, aggregate, and compare variants.
No retrieval or ranking execution.

In [ ]:
from pathlib import Path
import json
import sys
from collections import Counter

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.retrieval.evaluation.search_evaluation.failure_analysis_artifact import (
    validate_search_failure_analysis_artifact,
)
from productiq.retrieval.evaluation.search_evaluation.failure_analysis_runner import (
    run_default_search_failure_analysis,
    run_search_failure_analysis,
)
from productiq.retrieval.evaluation.search_evaluation.failure_analysis_schema import (
    SEARCH_FAILURE_ANALYSIS_FILENAME,
)

In [ ]:
result = run_search_failure_analysis(ROOT)
print("records:", len(result.records))
print("variants:", sorted({r.variant_name for r in result.records}))
print("K values:", sorted({r.analysis_k for r in result.records}))
print("lineage:", result.lineage.benchmark_name, result.lineage.benchmark_version)
for note in result.provenance_notes:
    print("note:", note)

In [ ]:
sample = [r for r in result.records if r.query_id == result.records[0].query_id][:8]
for row in sample:
    print(
        row.query_id,
        row.product_id,
        row.variant_name,
        f"K={row.analysis_k}",
        row.failure_category.value,
        f"rank={row.retrieval_rank}",
    )

In [ ]:
by_cat = Counter((a.variant_name, a.analysis_k, a.failure_category.value) for a in result.aggregates)
for key, count in sorted(by_cat.items())[:20]:
    print(key, "->", count)

In [ ]:
print("Retrieval relationships (BM25 x semantic, judged relevant):")
for row in result.retrieval_relationship_aggregates:
    print(row.pattern, row.product_count)

In [ ]:
ranking_rows = [r for r in result.records if r.rank_delta is not None]
print("rows with rank_delta:", len(ranking_rows))
for row in ranking_rows[:10]:
    print(
        row.query_id,
        row.product_id,
        row.variant_name,
        "ref",
        row.reference_rank,
        "cand",
        row.ranking_rank,
        "delta",
        row.rank_delta,
    )

In [ ]:
run_default_search_failure_analysis(ROOT)
path = ROOT / "resources" / "evaluation" / SEARCH_FAILURE_ANALYSIS_FILENAME
payload = json.loads(path.read_text(encoding="utf-8"))
validate_search_failure_analysis_artifact(payload)
print("checksum:", payload["deterministic_checksum_sha256"])
print("artifact:", path)